# Refinitiv Step1 Finalize Helper (Colab)

This notebook is for the `refinitiv_step1.zip` workflow:

- keep `refinitiv_step1.zip` in the Drive run root
- copy the zip into Colab local storage
- extract it into a temporary Colab workspace
- run `finalize_only` or `full` stage steps there
- copy either individual stage artifacts or the full updated `refinitiv_step1` tree back into Drive
- optionally refresh `refinitiv_step1.zip` on Drive from the updated Colab workspace

This notebook is intentionally scoped to the `refinitiv_step1` folder. It does not handle the separate `refinitiv_doc_ownership_lm2011` folder.

In [ ]:
from google.colab import drive
import os
import subprocess
import sys
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

IN_COLAB = 'google.colab' in sys.modules
REPO_ENV_VAR = 'NLP_THESIS_REPO_ROOT'
GIT_URL_ENV_VAR = 'NLP_THESIS_GIT_URL'
DEFAULT_GIT_URL = 'https://github.com/Erew42/NLP_Thesis.git'
CLONE_ROOT = Path('/content/NLP_Thesis')


def _resolve_drive_root() -> Path:
    for candidate in (
        Path('/content/drive/MyDrive'),
        Path('/content/drive/My Drive'),
        Path('/content/drive'),
    ):
        if candidate.exists():
            return candidate
    return Path('/content/drive')


def _find_repo_root(drive_root: Path) -> Path | None:
    override = os.environ.get(REPO_ENV_VAR, '').strip()
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend(
        [
            Path.cwd().resolve(),
            *Path.cwd().resolve().parents,
            CLONE_ROOT,
            drive_root / 'NLP_Thesis',
            drive_root / 'MyDrive' / 'NLP_Thesis',
            drive_root / 'My Drive' / 'NLP_Thesis',
        ]
    )
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'thesis_pkg' / 'pipeline.py').exists():
            return candidate
    return None


DRIVE_ROOT = _resolve_drive_root()
REPO_ROOT = _find_repo_root(DRIVE_ROOT)
if REPO_ROOT is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / 'src' / 'thesis_pkg' / 'pipeline.py').exists():
        raise FileExistsError(
            f'{CLONE_ROOT} exists but does not look like the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}.'
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', git_url, str(CLONE_ROOT)])
    REPO_ROOT = CLONE_ROOT

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the NLP_Thesis checkout. Set NLP_THESIS_REPO_ROOT or run this notebook in Colab so it can clone the repo automatically.'
    )

os.environ[REPO_ENV_VAR] = str(REPO_ROOT)
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DRIVE_WORK_ROOT = DRIVE_ROOT / 'Data_LM'
default_run_root = DRIVE_WORK_ROOT / 'results' / 'sec_ccm_unified_runner'
fallback_run_root = DRIVE_WORK_ROOT / 'full_data_run'
run_root_override = os.environ.get('REFINITIV_RUN_ROOT', '').strip()
DRIVE_RUN_ROOT = (
    Path(run_root_override).expanduser()
    if run_root_override
    else default_run_root
    if default_run_root.exists()
    else fallback_run_root
)
STEP1_ZIP_PATH = DRIVE_RUN_ROOT / 'refinitiv_step1.zip'
DRIVE_STEP1_DIR = DRIVE_RUN_ROOT / 'refinitiv_step1'
WORKSPACE_ROOT = Path('/content/refinitiv_step1_workspace')

print({
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'REPO_ROOT': str(REPO_ROOT),
    'DRIVE_RUN_ROOT': str(DRIVE_RUN_ROOT),
    'STEP1_ZIP_PATH': str(STEP1_ZIP_PATH),
    'DRIVE_STEP1_DIR': str(DRIVE_STEP1_DIR),
    'WORKSPACE_ROOT': str(WORKSPACE_ROOT),
})

In [ ]:
import shutil
import zipfile

from thesis_pkg.notebooks_and_scripts import refinitiv_local_api_runner as ref_runner

BATCH_PROFILE = 'local_safe'
STEP1_ONLY_STAGES = [
    'lookup_api',
    'resolution',
    'instrument_authority',
    'ownership_handoff',
    'ownership_api',
    'authority',
    'analyst_request_groups',
    'analyst_actuals_api',
    'analyst_estimates_api',
    'analyst_normalize',
]


def _find_workspace_run_root() -> Path:
    direct = WORKSPACE_ROOT
    if (direct / 'refinitiv_step1').exists():
        return direct
    nested = WORKSPACE_ROOT / 'full_data_run'
    if (nested / 'refinitiv_step1').exists():
        return nested
    matches = sorted(path.parent for path in WORKSPACE_ROOT.rglob('refinitiv_step1') if path.is_dir())
    if matches:
        return matches[0]
    raise FileNotFoundError('Could not find an extracted refinitiv_step1 folder in the workspace. Run prepare_workspace() first.')


def prepare_workspace(*, force_refresh: bool = False) -> Path:
    if not STEP1_ZIP_PATH.exists():
        raise FileNotFoundError(f'refinitiv_step1.zip not found: {STEP1_ZIP_PATH}')
    if force_refresh and WORKSPACE_ROOT.exists():
        shutil.rmtree(WORKSPACE_ROOT)
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    local_zip_path = WORKSPACE_ROOT / 'refinitiv_step1.zip'
    shutil.copy2(STEP1_ZIP_PATH, local_zip_path)
    with zipfile.ZipFile(local_zip_path, 'r') as zf:
        zf.extractall(WORKSPACE_ROOT)
    workspace_run_root = _find_workspace_run_root()
    print({'workspace_run_root': str(workspace_run_root)})
    return workspace_run_root


def _workspace_run_root() -> Path:
    return _find_workspace_run_root()


def _run_paths_for(run_root: Path):
    return ref_runner._resolve_run_paths(run_root, reviewed_ticker_allowlist_path=None)


def run_stage(stage: str, *, api_stage_mode: str = 'full', batch_profile: str = BATCH_PROFILE, extra_args: list[str] | None = None) -> int:
    workspace_run_root = _workspace_run_root()
    argv = [
        '--run-root',
        str(workspace_run_root),
        '--stage-start',
        stage,
        '--stage-stop',
        stage,
        '--api-stage-mode',
        api_stage_mode,
        '--batch-profile',
        batch_profile,
    ]
    if extra_args:
        argv.extend(extra_args)
    print('Running:', ' '.join(argv))
    exit_code = ref_runner.main(argv)
    print('Exit code:', exit_code)
    return exit_code


def _remove_path(path: Path) -> None:
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)
    elif path.exists() or path.is_symlink():
        path.unlink()


def _copy_path(src: Path, dst: Path) -> None:
    if not src.exists():
        print({'missing_source': str(src)})
        return
    if src.is_dir():
        if dst.exists() and not dst.is_dir():
            _remove_path(dst)
        if dst.exists():
            shutil.rmtree(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)
    else:
        if dst.exists() and dst.is_dir():
            shutil.rmtree(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print({'copied': str(src), 'to': str(dst)})


def sync_stage_back(stage: str, *, include_resume_state: bool = True) -> None:
    workspace_paths = _run_paths_for(_workspace_run_root())
    drive_paths = _run_paths_for(DRIVE_RUN_ROOT)
    workspace_outputs = ref_runner._stage_output_paths(workspace_paths, stage)
    drive_outputs = ref_runner._stage_output_paths(drive_paths, stage)
    copies: list[tuple[Path, Path]] = []
    for label, workspace_path in workspace_outputs.items():
        copies.append((workspace_path, drive_outputs[label]))
    copies.extend(zip(ref_runner._stage_manifest_outputs(workspace_paths, stage), ref_runner._stage_manifest_outputs(drive_paths, stage)))
    copies.extend(zip(ref_runner._stage_fetch_manifest_outputs(workspace_paths, stage), ref_runner._stage_fetch_manifest_outputs(drive_paths, stage)))
    if include_resume_state:
        copies.extend(zip(ref_runner._stage_resume_sentinels(workspace_paths, stage), ref_runner._stage_resume_sentinels(drive_paths, stage)))
    for src, dst in copies:
        _copy_path(src, dst)


def sync_step1_back() -> None:
    workspace_step1_dir = _workspace_run_root() / 'refinitiv_step1'
    if not workspace_step1_dir.exists():
        raise FileNotFoundError(f'workspace refinitiv_step1 not found: {workspace_step1_dir}')
    DRIVE_STEP1_DIR.mkdir(parents=True, exist_ok=True)
    copied_count = 0
    for src in sorted(workspace_step1_dir.iterdir()):
        dst = DRIVE_STEP1_DIR / src.name
        _copy_path(src, dst)
        copied_count += 1
    print({'synced_step1_entries': copied_count, 'drive_step1_dir': str(DRIVE_STEP1_DIR)})


def refresh_step1_zip_on_drive() -> Path:
    workspace_step1_dir = _workspace_run_root() / 'refinitiv_step1'
    if not workspace_step1_dir.exists():
        raise FileNotFoundError(f'workspace refinitiv_step1 not found: {workspace_step1_dir}')
    DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
    temp_zip_base = WORKSPACE_ROOT / 'refinitiv_step1_refreshed'
    temp_zip_path = temp_zip_base.with_suffix('.zip')
    if temp_zip_path.exists():
        temp_zip_path.unlink()
    created = Path(shutil.make_archive(str(temp_zip_base), 'zip', root_dir=_workspace_run_root(), base_dir='refinitiv_step1'))
    shutil.copy2(created, STEP1_ZIP_PATH)
    print({'refreshed_zip': str(STEP1_ZIP_PATH)})
    return STEP1_ZIP_PATH


def run_and_sync(stage: str, *, api_stage_mode: str = 'full', batch_profile: str = BATCH_PROFILE, extra_args: list[str] | None = None, include_resume_state: bool = True) -> int:
    exit_code = run_stage(stage, api_stage_mode=api_stage_mode, batch_profile=batch_profile, extra_args=extra_args)
    if exit_code == 0:
        sync_stage_back(stage, include_resume_state=include_resume_state)
    return exit_code


def show_stage_paths(stage: str) -> None:
    workspace_paths = _run_paths_for(_workspace_run_root())
    drive_paths = _run_paths_for(DRIVE_RUN_ROOT)
    print('workspace inputs:', ref_runner._stage_input_paths(workspace_paths, stage))
    print('workspace outputs:', ref_runner._stage_output_paths(workspace_paths, stage))
    print('drive outputs:', ref_runner._stage_output_paths(drive_paths, stage))
    print('workspace stage manifests:', ref_runner._stage_manifest_outputs(workspace_paths, stage))
    print('drive stage manifests:', ref_runner._stage_manifest_outputs(drive_paths, stage))
    print('workspace fetch manifests:', ref_runner._stage_fetch_manifest_outputs(workspace_paths, stage))
    print('drive fetch manifests:', ref_runner._stage_fetch_manifest_outputs(drive_paths, stage))


print('Supported stages in this zip-based notebook:', STEP1_ONLY_STAGES)

## Typical Sequence

1. Run `prepare_workspace(force_refresh=True)` after you have updated `refinitiv_step1.zip` on Drive.
2. Run stages inside the Colab workspace.
3. After each successful stage, either sync that stage back with `run_and_sync(...)` or mirror the whole updated workspace folder back with `sync_step1_back()`.
4. If you want the Drive zip to stay current as well, run `refresh_step1_zip_on_drive()` after syncing.

Typical order for the post-ownership part of step1 is:

1. `ownership_api` in `finalize_only`
2. `authority` in `full`
3. `analyst_request_groups` in `full`
4. `analyst_actuals_api` in `finalize_only`
5. `analyst_estimates_api` in `finalize_only`
6. `analyst_normalize` in `full`

If you refresh the zip on Drive with newer local staging, rerun `prepare_workspace(force_refresh=True)` before continuing. For the broadest roundtrip safety, prefer `sync_step1_back()` after a batch of Colab-side work because it also carries ledger/request/staging state back into the Drive folder.

In [ ]:
# Step 1: refresh the local Colab workspace from Drive.
# prepare_workspace(force_refresh=True)

# Optional inspection.
# show_stage_paths('ownership_api')

# Ownership finalize after local staging.
# run_and_sync('ownership_api', api_stage_mode='finalize_only')

# Downstream step1 stages.
# run_and_sync('authority')
# run_and_sync('analyst_request_groups')
# run_and_sync('analyst_actuals_api', api_stage_mode='finalize_only')
# run_and_sync('analyst_estimates_api', api_stage_mode='finalize_only')
# run_and_sync('analyst_normalize')

# Or, after several successful stages, sync the full updated step1 tree back to Drive.
# sync_step1_back()

# Optional: keep refinitiv_step1.zip on Drive aligned with the updated workspace.
# refresh_step1_zip_on_drive()